# **Replace the standard MLP in each GPT-2 transformer block with a sparse Mixture of Experts (MoE) layer and fine-tune on Alpace (instruct-reponse dataset)**

Import

In [24]:
import torch
import torch.nn as nn
import copy

from transformers import AutoTokenizer, GPT2LMHeadModel, TrainingArguments, Trainer, DataCollatorForSeq2Seq

from datasets import load_dataset

from google.colab import drive
import os


Config

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# specify save path
save_directory = "/content/drive/MyDrive/gpt2-moe-alpaca"

Mount Google Drive (to save/load model)

In [26]:
drive.mount('/content/drive')

# create the folder if it does not already exist
os.makedirs(save_directory, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Load base GPT2

In [37]:
# instantiate tokenizer
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')

# instantiate base pre-trained GPT2 model
model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [38]:
# print model architecture of base GPT2 model
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [39]:
# look at parameters under the hood in the base GPT2 model
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 124,439,808
Trainable Parameters: 124,439,808


In [40]:
# print parameters names and shapes from base GPT2 model
for name, param in model.named_parameters():
    print(f"{name:50} | Shape: {str(param.shape):30} | Trainable: {param.requires_grad}")

transformer.wte.weight                             | Shape: torch.Size([50257, 768])       | Trainable: True
transformer.wpe.weight                             | Shape: torch.Size([1024, 768])        | Trainable: True
transformer.h.0.ln_1.weight                        | Shape: torch.Size([768])              | Trainable: True
transformer.h.0.ln_1.bias                          | Shape: torch.Size([768])              | Trainable: True
transformer.h.0.attn.c_attn.weight                 | Shape: torch.Size([768, 2304])        | Trainable: True
transformer.h.0.attn.c_attn.bias                   | Shape: torch.Size([2304])             | Trainable: True
transformer.h.0.attn.c_proj.weight                 | Shape: torch.Size([768, 768])         | Trainable: True
transformer.h.0.attn.c_proj.bias                   | Shape: torch.Size([768])              | Trainable: True
transformer.h.0.ln_2.weight                        | Shape: torch.Size([768])              | Trainable: True
transformer.h.0.ln_

In [41]:
# # test code for direct parameter modification

# # isolate the specific parameters to change (e.g., first token embedding layer; see list above)
# embedding_weights = model.transformer.wte.weight

# # manually adjust parameters
# # use in-place operations with underscore suffix (e.g., .mul_(), .add_(), .copy_()) to update underlying PyTorch data
# with torch.no_grad(): # disable gradient calculation / computation graph
#     # multiply all weights by a specific scalar factor (e.g., 1.5)
#     embedding_weights.mul_(1.5)

#     # another example, manually set a specific parameter tensor to a custom tensor
#     # new_weights = torch.randn_like(embedding_weights)
#     # embedding_weights.copy_(new_weights)

In [42]:
# set base GPT2 model to evaluation mode
model.eval()

# tokenize input
inputs = tokenizer('Hello, my dog is cute', return_tensors='pt')

# generate token outputs from model
outputs = model.generate(**inputs, max_length=20)

# detokenize output into readable text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hello, my dog is cute. I'm not sure if she's a puppy or not. I


In each transformer block of GPT, replace MLP with router network plus 4 experts

In [43]:

# create custom PyTorch module for the GPT2 MoE Layer
# add a router network (selects the expert model)
# and 4 experts
#
# also have routing logging to see which experts are being called

class GPT2MoELayer(nn.Module):
    def __init__(self, original_mlp, num_experts=4):
        super().__init__()
        self.num_experts = num_experts

        # Extract hidden dimension
        hidden_dim = original_mlp.c_fc.weight.shape[0]

        # Router network and expert clones
        self.router = nn.Linear(hidden_dim, num_experts)
        self.experts = nn.ModuleList([copy.deepcopy(original_mlp) for _ in range(num_experts)])

        # Tracking telemetry (safe for training, defaults to off)
        self.routing_log = []
        self.track_routing = False

    def forward(self, hidden_states):
        orig_shape = hidden_states.shape
        flat_hidden_states = hidden_states.view(-1, orig_shape[-1])

        # Compute routing probabilities
        router_logits = self.router(flat_hidden_states)
        router_probs = torch.softmax(router_logits, dim=-1)

        # Select top-1 expert
        top1_weights, top1_indices = torch.topk(router_probs, k=1, dim=-1)

        # Log decisions if telemetry is explicitly turned on
        if self.track_routing:
            chosen_experts = top1_indices.squeeze(-1).tolist()
            if isinstance(chosen_experts, list):
                self.routing_log.extend(chosen_experts)
            else:
                self.routing_log.append(chosen_experts)

        # Route tokens to their designated expert
        flat_outputs = torch.zeros_like(flat_hidden_states)
        for expert_idx in range(self.num_experts):
            mask = (top1_indices.squeeze(-1) == expert_idx)

            if mask.any():
                expert_inputs = flat_hidden_states[mask]
                expert_outputs = self.experts[expert_idx](expert_inputs)
                flat_outputs[mask] = expert_outputs * top1_weights[mask]

        return flat_outputs.view(orig_shape)

In [44]:
# replace the MLPs in the transformer blocks

# Loop through each transformer layer and inject the MoE block
for i in range(len(model.transformer.h)):
    original_mlp = model.transformer.h[i].mlp

    # Replace the default MLP layer with your custom MoE router/expert layer
    model.transformer.h[i].mlp = GPT2MoELayer(original_mlp, num_experts=4)

In [45]:
# look at new architecture

# Print the updated architecture of the first block to verify the swap
print(model.transformer.h[0])

# Recalculate parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nNew Total Parameters: {total_params:,}")

GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MoELayer(
    (router): Linear(in_features=768, out_features=4, bias=True)
    (experts): ModuleList(
      (0-3): 4 x GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
)

New Total Parameters: 294,484,272


Fine-tune the new GPT2_MoE model with Alpaca (instruct and response)

In [46]:
# fine-tune the new GPT2_MoE model using Alpaca (instruct and response)

# Load the Alpaca dataset
dataset = load_dataset("tatsu-lab/alpaca")
# Split into train/validation for tracking performance
dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

In [47]:
# tokenize Alpaca dataset

# Configure padding token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def preprocess_alpaca(examples):
    prompts = []
    labels = []

    for i in range(len(examples["instruction"])):
        # Apply standard Alpaca formatting based on whether 'input' is provided
        if examples["input"][i]:
            prompt = (
                f"Below is an instruction that describes a task, paired with an input that provides further context. "
                f"Write a response that appropriately completes the request.\n\n"
                f"### Instruction:\n{examples['instruction'][i]}\n\n"
                f"### Input:\n{examples['input'][i]}\n\n"
                f"### Response:\n"
            )
        else:
            prompt = (
                f"Below is an instruction that describes a task. "
                f"Write a response that appropriately completes the request.\n\n"
                f"### Instruction:\n{examples['instruction'][i]}\n\n"
                f"### Response:\n"
            )

        response = examples["output"][i] + tokenizer.eos_token
        full_text = prompt + response

        # Tokenize prompt and full text separately to isolate lengths
        tokenized_prompt = tokenizer(prompt, truncation=True, max_length=512)["input_ids"]
        tokenized_full = tokenizer(full_text, truncation=True, max_length=512)["input_ids"]

        prompt_len = len(tokenized_prompt)

        # Construct label sequence: mask the prompt with -100, keep response token IDs
        label = [-100] * prompt_len + tokenized_full[prompt_len:]

        prompts.append(full_text)
        labels.append(label)

    # Re-tokenize everything uniformly with padding
    model_inputs = tokenizer(prompts, max_length=512, truncation=True, padding="max_length")

    # Pad the labels dynamically to match max_length
    padded_labels = []
    for l in labels:
        padded_l = l + [-100] * (512 - len(l))
        padded_labels.append(padded_l[:512])

    model_inputs["labels"] = padded_labels
    return model_inputs

# Map tokenization across entire dataset
tokenized_datasets = dataset.map(
    preprocess_alpaca,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/2601 [00:00<?, ? examples/s]

In [48]:
# training

# training parameters
training_args = TrainingArguments(
    output_dir="./gpt2-moe-alpaca",
    per_device_train_batch_size=4,      # Scale based on available VRAM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,     # Simulates a total batch size of 32
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    learning_rate=5e-5,                # Low learning rate preserves frozen-cloned layers
    weight_decay=0.01,
    warmup_ratio=0.03,                 # Essential for stabilizing new router layers
    logging_steps=10,
    num_train_epochs=1,                # 1 epoch is standard for Alpaca instruct tuning
    fp16=torch.cuda.is_available(),    # Enable mixed-precision if GPU supports it
    report_to="none"
)

# Use seq2seq data collator to cleanly package tensors
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Instantiate Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

# Set model back to train state and launch
model.train()
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
200,2.396268,2.214958
400,2.276430,2.118068
600,2.286313,2.083869
800,2.190556,2.051883
1000,2.134213,2.036069
1200,2.187584,2.024614
1400,2.106596,2.017629
1544,2.099293,2.015761


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1544, training_loss=2.312650315502147, metrics={'train_runtime': 1398.6579, 'train_samples_per_second': 35.32, 'train_steps_per_second': 1.104, 'total_flos': 3.871401376378061e+16, 'train_loss': 2.312650315502147, 'epoch': 1.0})

Save fine-tuned GPT2-MoE model to mounted Google Drive location

In [49]:
# save model to google drive

# save model and training configuration
trainer.save_model(save_directory)

# save tokenizer
tokenizer.save_pretrained(save_directory)

print(f"Fine-tuned MoE model has been saved to: {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned MoE model has been saved to: /content/drive/MyDrive/gpt2-moe-alpaca
